**What this code does:**
- Pulls data from NYC Open Data API (50k rows)
- Detects schema drift — new or dropped columns vs existing bronze table
- Adds audit columns — _loaddatetime, _batchid, _hashid etc.
- First run → creates and writes the table
- Subsequent runs → MERGE on _hashid to avoid duplicates
- Displays final table output
- API failure check




In [0]:
import requests
from pyspark.sql import functions as F
from datetime import datetime

try:
    url = "https://data.cityofnewyork.us/resource/m6nq-qud6.json"

    params = {
        "$limit" : 50000,
        "$offset": 0
    }

    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        raise Exception(f"API call failed with status: {response.status_code}")
    
    data = response.json()
    
    if not data:
        raise Exception("API returned empty data")
    
    df_newyorkdata = spark.createDataFrame(data)
    
  
    df_newyorkdata = df_newyorkdata.withColumn("test_new_col", F.lit("test"))
   

    source_schema = {}
    target_schema = {}

    # getting source data schema
    source_schema = dict(df_newyorkdata.dtypes)

    # getting target data schema
    if spark.catalog.tableExists("bronze.nyc_newyork"):
        tschema = dict(spark.table("bronze.nyc_newyork").dtypes)
        for col, dtype in tschema.items():
            if not col.startswith("_"):
                target_schema[col] = dtype

    add_col_source  = set(source_schema) - set(target_schema)
    drop_col_source = set(target_schema) - set(source_schema)

    # log added columns
    if add_col_source:
        for col in add_col_source:
            log_data = [{
                "log_datetime"  : datetime.now(),
                "table_name"    : "bronze.nyc_newyork",
                "drift_type"    : "added",
                "column_name"   : col,
                "batchid"       : datetime.now().strftime("%Y%m%d"),
                "source_system" : "nyc_open_data",
                "run_status"    : "success",
                "remarks"       : "new column detected"
            }]
            spark.createDataFrame(log_data).write.mode("append").saveAsTable("audit.schema_drift_log")
            for col in add_col_source:
                dtype = source_schema[col]
                dtype = "string" if dtype == "void" else dtype
                spark.sql(f"ALTER TABLE bronze.nyc_newyork ADD COLUMN {col} {dtype}")

        print(f"New columns added to source: {add_col_source}")
    
    # log dropped columns and add null to dataframe
    if drop_col_source:
        for col in drop_col_source:
            df_newyorkdata = df_newyorkdata.withColumn(col, F.lit(None))
            log_data = [{
                "log_datetime"  : datetime.now(),
                "table_name"    : "bronze.nyc_newyork",
                "drift_type"    : "dropped",
                "column_name"   : col,
                "batchid"       : datetime.now().strftime("%Y%m%d"),
                "source_system" : "nyc_open_data",
                "run_status"    : "success",
                "remarks"       : "column dropped from source"
            }]
            spark.createDataFrame(log_data).write.mode("append").saveAsTable("audit.schema_drift_log")
        print(f"Columns dropped from source: {drop_col_source}")

    hashid = F.concat_ws("", *[F.col(c) for c in df_newyorkdata.columns])

    df_add_col = (
        df_newyorkdata
        .withColumn("_loaddatetime", F.current_timestamp())
        .withColumn("_batchid",      F.date_format(F.current_date(), "yyyyMMdd"))
        .withColumn("_source_system",F.lit("nyc_open_data"))
        .withColumn("_load_count",   F.lit(df_newyorkdata.count()))
        .withColumn("_hashid",       F.md5(hashid))
    )

    if not spark.catalog.tableExists("bronze.nyc_newyork"):
        df_add_col.write.mode("overwrite").saveAsTable("bronze.nyc_newyork")
        print("Table created and data loaded")
    else:
        df_add_col.createOrReplaceTempView("vwSourceData")
        spark.sql("""
            MERGE INTO bronze.nyc_newyork AS target
            USING vwSourceData AS source
            ON target._hashid = source._hashid
            WHEN NOT MATCHED THEN INSERT *
        """)
        print("Merge completed")

    display(spark.sql("SELECT * FROM bronze.nyc_newyork"))

except Exception as e:
    print(e)